In [1]:
import os
import chromadb
from openai import OpenAI
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

In [2]:
#getting the api key
load_dotenv()
Api_key = os.getenv("OPENAI_API_KEY")
openai = OpenAI(api_key=Api_key)
#chroma client
client = chromadb.PersistentClient(path="./chromdb")

In [4]:
faqs = [
    {
        "id": "1",
        "question": "How do I reset my password?",
        "answer": "Click 'Forgot Password' on the login page and follow the instructions.",
        "category": "Account"
    },
    {
        "id": "2",
        "question": "How do I change my email address?",
        "answer": "Go to Settings > Profile and update your email.",
        "category": "Account"
    },
    {
        "id": "3",
        "question": "How do I cancel my subscription?",
        "answer": "Open Billing and click 'Cancel Subscription'.",
        "category": "Billing"
    },
    {
        "id": "4",
        "question": "How do I update my payment method?",
        "answer": "Go to Billing and choose 'Update Payment Method'.",
        "category": "Billing"
    },
    {
        "id": "5",
        "question": "Why can't I log in?",
        "answer": "Check your email, password, and internet connection. You can also reset your password.",
        "category": "Account"
    }
]

In [5]:
# OpenAI embedding function
embedding_function = OpenAIEmbeddingFunction(
    api_key=Api_key ,
    model_name="text-embedding-3-small"
)

In [26]:
#deleting old collection to add a new one since metadata does not appear
try:
    client.delete_collection("movies")
    print("Old collection deleted.")
except:
    print("Collection doesn't exist yet.")

Old collection deleted.


In [27]:
#create collection
collection = client.get_or_create_collection(
    name ="FAQs",
    embedding_function=embedding_function
)

In [28]:
#loading data
metadatas = []
documents = []
ids = []

for faq in faqs:
    documents.append(
        f"""
        Question: {faq['question']}
        Answer: {faq['answer']}
        """
    )

    ids.append(faq["id"])

    metadatas.append({
        "question": faq["question"],
        "answer": faq["answer"],
        "category": faq["category"]
    })



In [29]:
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
    )

In [34]:
result = collection.query(
    query_texts=['Im not able to log in'],
    n_results=1
)

In [35]:
print("ID:", result["ids"][0][0])
print("Question:", result["documents"][0][0])


print("Distance:", result["distances"][0][0])

ID: 5
Question: 
        Question: Why can't I log in?
        Answer: Check your email, password, and internet connection. You can also reset your password.
        
Distance: 0.46764588356018066
